Without pydantic 
we can add any type of data into any field that causes error in model later

In [4]:
def add_patient_data(name,age):
    print(name)
    print(age)
    print('data saved sucessfully')

add_patient_data(21,12)

21
12
data saved sucessfully


With pydantic we can restrict the data types so only correct type of value can go into the correct variable 

In [29]:
def add_patient_data(name:str,age:int):
    if type(name)== str and type(age)== int:
        print(name)
        print(age)
        print('data saved sucessfully!')
    else:
        raise TypeError('check the types for name and age')

add_patient_data('pratik',12)

pratik
12
data saved sucessfully!


In [62]:
from pydantic import BaseModel

class patient_data_model(BaseModel):
    name: str
    age: int

data = {"name":"smith","age":12}

patient1 = patient_data_model(**data)

def add_data(patient1: patient_data_model):
    print(patient1.model_dump())
  

add_data(patient1)

{'name': 'smith', 'age': 12}


Data validation

In [64]:
from pydantic import BaseModel,EmailStr,AnyUrl,Field
from typing import List,Dict,Optional,Annotated #to provide the metadata use Annotated

class patient_data_model(BaseModel):
    name: str = Field(max_length=20)
    age: int = Field(gt=1,lt=100)
    email: EmailStr
    linkedin: AnyUrl
    allergies: Optional[ List[str] ] = None
    is_married: Annotated [bool, Field(description='Marital status',examples='Married,Unmarried')]
    contact_details: Dict[str,str]

data = {"name":"john","age":34,"email":"abc@mail.com","allergies":["penuts","mashrooms"],"is_married":"False","linkedin":"https://www.google.com","contact_details": {"email":"john@mail.com","mob":"121212"}}
patient1 = patient_data_model(**data)

def displaydata(patient1:patient_data_model):
    print(patient1.model_dump())

displaydata(patient1)

{'name': 'john', 'age': 34, 'email': 'abc@mail.com', 'linkedin': AnyUrl('https://www.google.com/'), 'allergies': ['penuts', 'mashrooms'], 'is_married': False, 'contact_details': {'email': 'john@mail.com', 'mob': '121212'}}


Field Validator

In [94]:
from pydantic import BaseModel,field_validator

class patient_data_model(BaseModel):
    name: str
    age: int
    email: str

    #=======================================================================================================================
    @field_validator('email') #which field we want to validate 
    @classmethod #this is the method of patient_data_model
    def validate_email(cls,value): #cls= class , value = email

        valid_domains = ['icici.com','hdfc.com']
        input_domains = value.split('@')[-1] #split the email ex - pratik@mail.com = mail.com

        if input_domains not in valid_domains: #check input email with valid email domains mail.com != icici.com
            raise ValueError('Not a valid domain') #raise error
        return value
    #=======================================================================================================================
    @field_validator('name')
    @classmethod
    def uppercase_name(cls,value):
        return value.upper()
    #=======================================================================================================================
    

data = {"name":"smith","age":12,"email":'pratik@icici.com'}

patient1 = patient_data_model(**data)

def add_data(patient1: patient_data_model):
    print(patient1.model_dump())
  

add_data(patient1)

{'name': 'SMITH', 'age': 12, 'email': 'pratik@icici.com'}



```markdown
## Model Validator

**Model validators** can access **all fields** in the model, unlike **field validators**, which handle only a single field.

### Modes

Model validators support two modes:

- `before`
- `after`

#### `after` mode
- Runs **after** field-level validation.
- Use when you need to:
  - Cross-check multiple fields.
  - Apply logical or business-rule validation across the model.

#### `before` mode
- Runs **before** field-level validation.
- Use when you want to:
  - Modify raw input.
  - Perform data transformation before normal validation happens.
```

In [110]:
from pydantic import BaseModel,model_validator

class patient_data_model(BaseModel):
    name: str
    age: int
    emergency_contact: Dict[str,str]

    @model_validator(mode='after') 
    @classmethod

    def validate_patient_age_emergency_contact(cls,model):
        if model.age >= 60 and 'emergency' not in model.emergency_contact: #here we can access all model fileds 
            raise ValueError('Patients having age above 60 must provide emergency contact details')
        return model


data = {"name":"josheph","age": 60,"emergency_contact": {"phone":"121212","emergency":"122323"}}

def Add_patient_data(patient1:patient_data_model):
    print(patient1.model_dump())

patient1 = patient_data_model(**data)
Add_patient_data(patient1)


{'name': 'josheph', 'age': 60, 'emergency_contact': {'phone': '121212', 'emergency': '122323'}}


/var/folders/z8/cxl79vz558q3_ts3_l1n5zj00000gn/T/ipykernel_7706/93737395.py:8: PydanticDeprecatedSince212: Using `@model_validator` with mode='after' on a classmethod is deprecated. Instead, use an instance method. See the documentation at https://docs.pydantic.dev/2.13/concepts/validators/#model-after-validator. Deprecated in Pydantic V2.12 to be removed in V3.0.
  @model_validator(mode='after')


Computed Fields - used when we want to compute the data on the go 

In [113]:
from pydantic import BaseModel,computed_field

class patient_data_model(BaseModel):
    name: str
    age: int
    height: float
    weight: float
    #no need to define the computed fields as we are computing on the go 

    @computed_field
    @property

    def calculated_bmi(self)-> float:
        bmi = round(self.weight / (self.height**2),2)
        
        return bmi


data = {"name":"joohn","age": 20,"height":1.75,"weight":50}
patient1 = patient_data_model(**data)

def add_patient_data(patient:patient_data_model):
    print('BMI = ', patient.calculated_bmi) # for computed field access field with function

add_patient_data(patient1)





BMI =  16.33


Nested Models - used when we want to use complex fields like address - city,state,pin,address_line1,address_line2

In [ ]:
from pydantic import BaseModel

class address_model(BaseModel):
    state: str
    city: str 
    pin: int 

class patient_data_model(BaseModel):
    name : str
    age : int 
    gender: str 
    address: address_model

 
address_data = {"state": "maharastra","city":"nashik","pin":1212}
address1 = address_model(**address_data)

patient_data = {"name": "johnn","age":23,"gender":"male","address":address1}
patient1 = patient_data_model(**patient_data)

def add_patient_data(patient:patient_data_model):
    print(patient.address)

add_patient_data(patient1)



Serialization - export the validated data in json 

In [ ]:
from pydantic import BaseModel

class address_model(BaseModel):
    city: str
    state: str
    pin: int 

class patient_data_model(BaseModel):
    name : str
    age : int 
    gender: str 
    address: address_model


address_data = {"city":"nashik","state":"maharshtra","pin":1212}
address1 = address_model(**address_data)

patient_data = {"name":"john","age":12,"gender":"male","address":address1}
patient1 = patient_data_model(**patient_data)

print(patient1.model_dump_json()) #dumping data in json format 


{"name":"john","age":12,"gender":"male","address":{"city":"nashik","state":"maharshtra","pin":1212}}
